# AI 620 - Assignment 3, Part 1
Data Quality and Validation in Real-world Datasets


Dependencies


In [1]:
import subprocess, sys

packages = [
    "psycopg2-binary",
    "pandas",
    "numpy",
    "great-expectations==0.18.22",
    "faker",
    "ipywidgets",
]

for pkg in packages:
    print(f"Installing {pkg}...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "-q"],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f" {result.stderr.strip()}")
    else:
        print(f" done")

print("\nAll packages installed.")


Installing psycopg2-binary...
 done
Installing pandas...
 done
Installing numpy...
 done
Installing great-expectations==0.18.22...
 done
Installing faker...
 done
Installing ipywidgets...
 done

All packages installed.


## Task 1  Generate Synthetic Datasets 

In [2]:
import subprocess, os, time, sys

# ── PostgreSQL setup ───────────────────────────────────
def ensure_postgres():
    pg_ctl_candidates = [
        "/opt/homebrew/opt/postgresql@15/bin/pg_ctl",
        "/opt/homebrew/opt/postgresql@16/bin/pg_ctl",
        "/opt/homebrew/bin/pg_ctl",
    ]
    psql_candidates = [
        "/opt/homebrew/opt/postgresql@15/bin/psql",
        "/opt/homebrew/opt/postgresql@16/bin/psql",
        "/opt/homebrew/bin/psql",
    ]

    pg_ctl = next((p for p in pg_ctl_candidates if os.path.exists(p)), None)
    psql   = next((p for p in psql_candidates   if os.path.exists(p)), None)

    if psql is None:
        raise EnvironmentError(
            "psql not found. Install PostgreSQL first:\n"
            "  brew install postgresql@15"
        )

    try:
        result = subprocess.run([psql, "-U", os.environ.get("USER","postgres"),
                                  "-c", "SELECT 1;", "postgres"],
                                 capture_output=True, timeout=5)
        if result.returncode == 0:
            print("PostgreSQL already running.")
            return psql
    except Exception:
        pass

    if pg_ctl:
        data_dir_candidates = [
            f"/opt/homebrew/var/postgresql@15",
            f"/opt/homebrew/var/postgresql@16",
            f"/opt/homebrew/var/postgres",
        ]
        data_dir = next((d for d in data_dir_candidates if os.path.isdir(d)), None)
        if data_dir:
            print("🔄 Starting PostgreSQL...")
            subprocess.run([pg_ctl, "-D", data_dir, "start", "-l", "/tmp/pg.log"],
                           capture_output=True)
            time.sleep(3)
            print("PostgreSQL started.")
    return psql

PSQL = ensure_postgres()
print(f"   psql binary: {PSQL}")


PostgreSQL already running.
   psql binary: /opt/homebrew/opt/postgresql@15/bin/psql


In [3]:
import subprocess, os, textwrap

PSQL  = PSQL          # from previous cell
USER  = os.environ.get("USER", "postgres")
OUTDIR = os.getcwd() 
os.makedirs(OUTDIR, exist_ok=True)


CLEAN_CSV     = os.path.join(OUTDIR, "clean_synthetic_dataset.csv")
CORRUPTED_CSV = os.path.join(OUTDIR, "corrupted_synthetic_dataset.csv")

def run_psql(sql: str, db: str = "postgres") -> str:
    """Run arbitrary SQL via psql subprocess and return stdout."""
    result = subprocess.run(
        [PSQL, "-U", USER, "-d", db, "-c", sql],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        if "already exists" not in result.stderr:
            print(f"  ⚠️  SQL warning: {result.stderr.strip()[:300]}")
    return result.stdout

# ── Create DATABASE ────────────────────────────-─────────────────────
print("Creating database...")
run_psql("CREATE DATABASE pakwheels_db;")
print("pakwheels_db ready")

# ── Create clean table ─────────────────────────────────────────────────────────
print("Creating schema...")
run_psql("""
    DROP TABLE IF EXISTS pakwheels_clean;
    CREATE TABLE pakwheels_clean (
        id           SERIAL PRIMARY KEY,
        make         VARCHAR(50),
        model        VARCHAR(50),
        year         INT,
        engine_cc    INT,
        mileage_km   INT,
        city         VARCHAR(50),
        body_type    VARCHAR(30),
        transmission VARCHAR(20),
        fuel_type    VARCHAR(20),
        price_pkr    BIGINT,
        color        VARCHAR(30),
        registered   BOOLEAN,
        num_owners   INT
    );
""", db="pakwheels_db")
print("Schema created")

# ── Insert 50000 synthetic rows ────────────────────────────────────────
print("Inserting 50,000 clean rows ...")
run_psql("""
    INSERT INTO pakwheels_clean (
        make, model, year, engine_cc, mileage_km,
        city, body_type, transmission, fuel_type,
        price_pkr, color, registered, num_owners
    )
    SELECT
        (ARRAY['Suzuki','Toyota','Honda','Kia','Hyundai','Daihatsu','Mitsubishi'])[floor(random()*7)+1],
        (ARRAY['Alto','Cultus','Corolla','City','Civic','Sportage','Tucson','Mira','Lancer'])[floor(random()*9)+1],
        2005 + floor(random()*19)::int,
        (ARRAY[660,800,1000,1300,1500,1600,1800,2000,2400])[floor(random()*9)+1],
        floor(random()*200000 + 5000)::int,
        (ARRAY['Karachi','Lahore','Islamabad','Peshawar','Quetta','Multan','Faisalabad'])[floor(random()*7)+1],
        (ARRAY['Sedan','Hatchback','SUV','Crossover','MPV','Pickup'])[floor(random()*6)+1],
        (ARRAY['Automatic','Manual'])[floor(random()*2)+1],
        (ARRAY['Petrol','Diesel','Hybrid','CNG'])[floor(random()*4)+1],
        floor(random()*4500000 + 500000)::bigint,
        (ARRAY['White','Silver','Black','Red','Blue','Grey','Brown','Green'])[floor(random()*8)+1],
        (random() > 0.1),
        floor(random()*4 + 1)::int
    FROM generate_series(1, 50000);
""", db="pakwheels_db")
print("50,000 clean rows inserted")

print("Exporting clean CSV...")
run_psql(f"\\COPY pakwheels_clean TO '{CLEAN_CSV}' WITH CSV HEADER;", db="pakwheels_db")
print(f"Exported : {CLEAN_CSV}")


Creating database...
pakwheels_db ready
Creating schema...
Schema created
Inserting 50,000 clean rows ...
50,000 clean rows inserted
Exporting clean CSV...
Exported : /Users/nifrawahaj/UNI/sem2/ai620/PA/PA3/25280002_AI620_ASSIGNMENT3_Part_1/clean_synthetic_dataset.csv


In [4]:
# ── Create CORRUPTED table ─────────────────────────────────────────────────────
print("Creating corrupted table (copy of clean)...")
run_psql("""
    DROP TABLE IF EXISTS pakwheels_corrupted;
    CREATE TABLE pakwheels_corrupted AS TABLE pakwheels_clean;
""", db="pakwheels_db")

corruptions = [
    ("NULL ~8% of engine_cc",
     "UPDATE pakwheels_corrupted SET engine_cc = NULL WHERE random() < 0.08;"),

    ("NULL ~6% of mileage_km",
     "UPDATE pakwheels_corrupted SET mileage_km = NULL WHERE random() < 0.06;"),

    ("NULL ~4% of city",
     "UPDATE pakwheels_corrupted SET city = NULL WHERE random() < 0.04;"),

    ("Impossible years (1800/1950/2035/2099)",
     "UPDATE pakwheels_corrupted SET year = (ARRAY[1800,1950,2035,2099])[floor(random()*4)+1] WHERE random() < 0.04;"),

    ("Negative prices",
     "UPDATE pakwheels_corrupted SET price_pkr = -price_pkr WHERE random() < 0.03;"),

    ("Out-of-range engine_cc (10k-60k)",
     "UPDATE pakwheels_corrupted SET engine_cc = floor(random()*50000 + 10000)::int WHERE random() < 0.03;"),

    ("Misspelled transmission values",
     "UPDATE pakwheels_corrupted SET transmission = (ARRAY['Manuall','AUTO','autmatic','MANUAL '])[floor(random()*4)+1] WHERE random() < 0.05;"),

    ("Misspelled fuel_type values",
     "UPDATE pakwheels_corrupted SET fuel_type = (ARRAY['Gazoline','PETROL','deisel','Gas'])[floor(random()*4)+1] WHERE random() < 0.05;"),

    ("Insert 500 duplicate rows (PK violations in flat file)",
     """INSERT INTO pakwheels_corrupted
        SELECT id, make, model, year, engine_cc, mileage_km, city, body_type,
               transmission, fuel_type, price_pkr, color, registered, num_owners
        FROM pakwheels_corrupted ORDER BY random() LIMIT 500;"""),
]

for desc, sql in corruptions:
    run_psql(sql, db="pakwheels_db")
    print(f"Corruption applied: {desc}")

# Export corrupted CSV
print("\nExporting corrupted CSV...")
run_psql(f"\\COPY pakwheels_corrupted TO '{CORRUPTED_CSV}' WITH CSV HEADER;", db="pakwheels_db")
print(f" Exported : {CORRUPTED_CSV}")


Creating corrupted table (copy of clean)...
Corruption applied: NULL ~8% of engine_cc
Corruption applied: NULL ~6% of mileage_km
Corruption applied: NULL ~4% of city
Corruption applied: Impossible years (1800/1950/2035/2099)
Corruption applied: Negative prices
Corruption applied: Out-of-range engine_cc (10k-60k)
Corruption applied: Misspelled transmission values
Corruption applied: Misspelled fuel_type values
Corruption applied: Insert 500 duplicate rows (PK violations in flat file)

Exporting corrupted CSV...
 Exported : /Users/nifrawahaj/UNI/sem2/ai620/PA/PA3/25280002_AI620_ASSIGNMENT3_Part_1/corrupted_synthetic_dataset.csv


In [5]:
import pandas as pd

clean_df     = pd.read_csv(CLEAN_CSV)
corrupted_df = pd.read_csv(CORRUPTED_CSV)

print(f"Clean dataset     : {clean_df.shape[0]:,} rows × {clean_df.shape[1]} columns")
print(f"Corrupted dataset : {corrupted_df.shape[0]:,} rows × {corrupted_df.shape[1]} columns")
print()
print("=== Clean — first 3 rows ===")
display(clean_df.head(3))
print()
print("=== Corrupted — first 3 rows ===")
display(corrupted_df.head(3))


Clean dataset     : 50,000 rows × 14 columns
Corrupted dataset : 50,500 rows × 14 columns

=== Clean — first 3 rows ===


,id,make,model,year,engine_cc,mileage_km,city,body_type,transmission,fuel_type,price_pkr,color,registered,num_owners
0,1,Hyundai,Alto,2023,1000,51867,Karachi,Hatchback,Automatic,CNG,1324397,Red,t,1
1,2,Mitsubishi,Lancer,2022,1600,189626,Faisalabad,Sedan,Manual,Hybrid,2407186,Blue,t,3
2,3,Suzuki,Alto,2007,1500,203861,Multan,Pickup,Automatic,Diesel,1576051,Brown,t,1



=== Corrupted — first 3 rows ===


,id,make,model,year,engine_cc,mileage_km,city,body_type,transmission,fuel_type,price_pkr,color,registered,num_owners
0,2,Mitsubishi,Lancer,2022,1600.0,189626.0,Faisalabad,Sedan,Manual,Hybrid,2407186,Blue,t,3
1,3,Suzuki,Alto,2007,1500.0,203861.0,Multan,Pickup,Automatic,Diesel,1576051,Brown,t,1
2,5,Daihatsu,Mira,2017,2000.0,58871.0,Faisalabad,Sedan,Automatic,Petrol,976863,Grey,t,3


In [6]:
print("=== NULL counts: Clean vs Corrupted ===")
null_compare = pd.DataFrame({
    "clean_nulls"    : clean_df.isnull().sum(),
    "corrupted_nulls": corrupted_df.isnull().sum(),
})
null_compare["difference"] = null_compare["corrupted_nulls"] - null_compare["clean_nulls"]
display(null_compare)


=== NULL counts: Clean vs Corrupted ===


,clean_nulls,corrupted_nulls,difference
id,0,0,0
make,0,0,0
model,0,0,0
year,0,0,0
engine_cc,0,3951,3951
mileage_km,0,3015,3015
city,0,2011,2011
body_type,0,0,0
transmission,0,0,0
fuel_type,0,0,0


---
## Task 2 Great Expectations, Data Quality Validation

In [7]:
import great_expectations as gx
from great_expectations.core.batch import RuntimeBatchRequest
import os, json, warnings
warnings.filterwarnings("ignore")

GX_DIR = os.path.join(OUTDIR, "gx")
os.makedirs(GX_DIR, exist_ok=True)

context = gx.get_context(context_root_dir=GX_DIR)
print(f"Great Expectations context at: {GX_DIR}")
print(f"   GX version: {gx.__version__}")


Great Expectations context at: /Users/nifrawahaj/UNI/sem2/ai620/PA/PA3/25280002_AI620_ASSIGNMENT3_Part_1/gx
   GX version: 0.18.22


In [8]:
DATASOURCE_NAME = "pakwheels_pandas"

try:
    datasource = context.get_datasource(DATASOURCE_NAME)
    print(f"Datasource '{DATASOURCE_NAME}' already exists.")
except Exception:
    datasource = context.sources.add_pandas(name=DATASOURCE_NAME)
    print(f"Datasource '{DATASOURCE_NAME}' created.")


Datasource 'pakwheels_pandas' already exists.


In [9]:
# ── Build Expectation Suite ────────────────────────────────────────────────
SUITE_NAME = "PakWheels Data Quality Validation"

suite = context.add_or_update_expectation_suite(expectation_suite_name=SUITE_NAME)

# Helper shorthand
def add(exp_type, **kwargs):
    from great_expectations.core import ExpectationConfiguration
    suite.add_expectation(ExpectationConfiguration(expectation_type=exp_type, kwargs=kwargs))

# ──────────────────────────── Schema ─────────────────────────────────────
add("expect_table_columns_to_match_set",
    column_set=["id","make","model","year","engine_cc","mileage_km",
                "city","body_type","transmission","fuel_type",
                "price_pkr","color","registered","num_owners"],
    exact_match=False)

# ── Row count sanity ────────────────────────────────────────────────────────
add("expect_table_row_count_to_be_between", min_value=1000, max_value=200000)

# ── Uniqueness, id must not duplicate ──────────────────────────────────────
add("expect_column_values_to_be_unique", column="id")

# ── Completeness, critical columns not be null ────────────────────────
for col in ["id","make","model","year","price_pkr","transmission","fuel_type"]:
    add("expect_column_values_to_not_be_null", column=col, mostly=0.99)

# ── Validity, year in plausible range ──────────────────────────────────────
add("expect_column_values_to_be_between",
    column="year", min_value=1990, max_value=2025, mostly=0.97)

# ── Validity engine_cc  ─────────────────────────────────────────────────
add("expect_column_values_to_be_between",
    column="engine_cc", min_value=500, max_value=5000, mostly=0.95)

# ── Validity, mileage non-negative and realistic ───────────────────────────
add("expect_column_values_to_be_between",
    column="mileage_km", min_value=0, max_value=1000000, mostly=0.97)

# ── Validity: price must be positive and in market range ───────────────────
add("expect_column_values_to_be_between",
    column="price_pkr", min_value=100000, max_value=50000000, mostly=0.97)

# ── Validity: num_owners realistic ─────────────────────────────────────────
add("expect_column_values_to_be_between",
    column="num_owners", min_value=1, max_value=10, mostly=0.98)

# ── Categorical: transmission ─────────────────────────────────────────────
add("expect_column_values_to_be_in_set",
    column="transmission", value_set=["Automatic","Manual"], mostly=0.95)

# ── Categorical: fuel_type ────────────────────────────────────────────────
add("expect_column_values_to_be_in_set",
    column="fuel_type", value_set=["Petrol","Diesel","Hybrid","CNG"], mostly=0.95)

# ── Categorical: body_type ────────────────────────────────────────────────
add("expect_column_values_to_be_in_set",
    column="body_type",
    value_set=["Sedan","Hatchback","SUV","Crossover","MPV","Pickup"],
    mostly=0.97)

# ── Distributional: median price ──────────────────────────────────────────
add("expect_column_median_to_be_between",
    column="price_pkr", min_value=500000, max_value=10000000)

# ── Distributional: mean engine_cc ────────────────────────────────────────
add("expect_column_mean_to_be_between",
    column="engine_cc", min_value=800, max_value=2500)

# ── Type: id should be integer ────────────────────────────────────────────
add("expect_column_values_to_be_of_type", column="id", type_="int64")


add("expect_compound_columns_to_be_unique",
    column_list=["make", "model", "year", "price_pkr"])

add("expect_column_values_to_be_between",
    column="price_pkr",
    min_value=100000,
    max_value=50000000,
    meta={"section": "Validity - Price"})

context.update_expectation_suite(expectation_suite=suite)
print(f"Expectation Suite '{SUITE_NAME}' saved with {len(suite.expectations)} expectations.")


Expectation Suite 'PakWheels Data Quality Validation' saved with 22 expectations.


In [10]:
def validate_dataframe(df: "pd.DataFrame", label: str) -> dict:
    """
    Loads df as a GX in-memory batch, runs the suite, prints a summary,
    saves a JSON result, and returns the results dict.
    """
    import pandas as pd

    asset_name = f"{label}_asset"
    try:
        asset = datasource.get_asset(asset_name)
    except Exception:
        asset = datasource.add_dataframe_asset(name=asset_name)

    batch_request = asset.build_batch_request(dataframe=df)

    validator = context.get_validator(
        batch_request=batch_request,
        expectation_suite_name=SUITE_NAME,
    )

    results = validator.validate()
    stats   = results["statistics"]

    passed = stats["successful_expectations"]
    failed = stats["unsuccessful_expectations"]
    total  = stats["evaluated_expectations"]
    pct    = stats["success_percent"]

    bar_p = "█" * int(pct / 5)
    bar_f = "░" * (20 - int(pct / 5))

    print(f"\n{'='*55}")
    print(f"  Dataset : {label}")
    print(f"  Rows    : {df.shape[0]:,}  |  Columns: {df.shape[1]}")
    print(f"  {'='*47}")
    print(f"  Total expectations : {total}")
    print(f"  Passed           : {passed}")
    print(f"  Failed           : {failed}")
    print(f"  Success rate        : [{bar_p}{bar_f}] {pct:.1f}%")
    print(f"{'='*55}")

    if failed > 0:
        print("  Failed expectations:")
        for r in results["results"]:
            if not r["success"]:
                exp = r["expectation_config"]["expectation_type"]
                col = r["expectation_config"]["kwargs"].get("column","(table)")
                print(f"    ✗  {exp}  →  column: {col}")

    # Save JSON
    json_path = os.path.join(OUTDIR, f"validation_{label}.json")
    with open(json_path, "w") as f:
        json.dump(results.to_json_dict(), f, indent=2)
    print(f"\n  📄 JSON report → {json_path}")

    return results

print("Validation function ready.")


Validation function ready.


In [11]:
# ── Run validation on CLEAN dataset ───────────────────────────────────────────
clean_results = validate_dataframe(clean_df, "clean_dataset")


Calculating Metrics:   0%|          | 0/65 [00:00<?, ?it/s]


  Dataset : clean_dataset
  Rows    : 50,000  |  Columns: 14
  Total expectations : 22
  Passed           : 22
  Failed           : 0
  Success rate        : [████████████████████] 100.0%

  📄 JSON report → /Users/nifrawahaj/UNI/sem2/ai620/PA/PA3/25280002_AI620_ASSIGNMENT3_Part_1/validation_clean_dataset.json


In [12]:
# ── Run validation on CORRUPTED dataset ───────────────────────────────────────
corrupted_results = validate_dataframe(corrupted_df, "corrupted_dataset")


Calculating Metrics:   0%|          | 0/65 [00:00<?, ?it/s]


  Dataset : corrupted_dataset
  Rows    : 50,500  |  Columns: 14
  Total expectations : 22
  Passed           : 15
  Failed           : 7
  Success rate        : [█████████████░░░░░░░] 68.2%
  Failed expectations:
    ✗  expect_compound_columns_to_be_unique  →  column: (table)
    ✗  expect_column_values_to_be_unique  →  column: id
    ✗  expect_column_values_to_be_between  →  column: year
    ✗  expect_column_values_to_be_between  →  column: price_pkr
    ✗  expect_column_values_to_be_in_set  →  column: transmission
    ✗  expect_column_values_to_be_in_set  →  column: fuel_type
    ✗  expect_column_mean_to_be_between  →  column: engine_cc

  📄 JSON report → /Users/nifrawahaj/UNI/sem2/ai620/PA/PA3/25280002_AI620_ASSIGNMENT3_Part_1/validation_corrupted_dataset.json


In [13]:
# ── Generate HTML Data Docs ────────────────────────────────────────────────────
context.build_data_docs()
docs_path = os.path.join(GX_DIR, "uncommitted", "data_docs", "local_site", "index.html")
print(f"HTML Data Docs generated {docs_path}")

HTML Data Docs generated /Users/nifrawahaj/UNI/sem2/ai620/PA/PA3/25280002_AI620_ASSIGNMENT3_Part_1/gx/uncommitted/data_docs/local_site/index.html


Comparison Summary

In [14]:
import pandas as pd

def extract_stats(results, label):
    rows = []
    for r in results["results"]:
        exp  = r["expectation_config"]["expectation_type"]
        col  = r["expectation_config"]["kwargs"].get("column", "(table-level)")
        succ = "✅ PASS" if r["success"] else "❌ FAIL"
        rows.append({"Expectation": exp, "Column": col, label: succ})
    return pd.DataFrame(rows).set_index(["Expectation","Column"])

clean_tbl     = extract_stats(clean_results,     "Clean")
corrupted_tbl = extract_stats(corrupted_results, "Corrupted")

comparison = clean_tbl.join(corrupted_tbl, how="outer")
display(comparison.style.applymap(
    lambda v: "background-color:#d4edda" if "PASS" in str(v)
    else ("background-color:#f8d7da" if "FAIL" in str(v) else ""),
))


In [15]:
cs = clean_results["statistics"]
xs = corrupted_results["statistics"]

summary = pd.DataFrame({
    "Metric": [
        "Total Expectations",
        "Passed",
        "Failed",
        "Success Rate (%)",
    ],
    "Clean Dataset": [
        cs["evaluated_expectations"],
        cs["successful_expectations"],
        cs["unsuccessful_expectations"],
        f"{cs['success_percent']:.1f}%",
    ],
    "Corrupted Dataset": [
        xs["evaluated_expectations"],
        xs["successful_expectations"],
        xs["unsuccessful_expectations"],
        f"{xs['success_percent']:.1f}%",
    ],
})
display(summary)


,Metric,Clean Dataset,Corrupted Dataset
0,Total Expectations,22,22
1,Passed,22,15
2,Failed,0,7
3,Success Rate (%),100.0%,68.2%


# Task 3 Data Quality Report with Great Expectations on Pak-Wheels Datase

In [16]:
import pandas as pd
import great_expectations as gx
from great_expectations.core.expectation_configuration import ExpectationConfiguration

context = gx.get_context(context_root_dir=GX_DIR) 
df_pak = pd.read_csv("real_workd_pakwheels.csv")

print(f"Dataset loaded: {df_pak.shape[0]} rows, {df_pak.shape[1]} columns.")

Dataset loaded: 62302 rows, 14 columns.


In [17]:
SUITE_NAME = "PakWheels_Real_World_Validation"

suite = context.add_or_update_expectation_suite(expectation_suite_name=SUITE_NAME)

def add_exp(exp_type, **kwargs):
    suite.add_expectation(
        ExpectationConfiguration(expectation_type=exp_type, kwargs=kwargs)
    )


In [18]:
add("expect_table_columns_to_match_set",
    column_set=[
        "addref","city","body","make","model",
        "year","engine","transmission","fuel",
        "mileage","price"
    ],
    exact_match=False
)
add("expect_table_row_count_to_be_between",
    min_value=100, max_value=500000
)

for col in ["addref","year","engine","mileage","price","transmission","fuel"]:
    add("expect_column_values_to_not_be_null",
        column=col,
        mostly=0.95
    )

add("expect_column_values_to_be_unique", column="addref")

add("expect_column_values_to_be_between",
    column="year", min_value=1990, max_value=2026, mostly=0.98
)

add("expect_column_values_to_be_between",
    column="engine", min_value=500, max_value=6000, mostly=0.95
)

add("expect_column_values_to_be_between",
    column="mileage", min_value=0, max_value=500000, mostly=0.95
)

add("expect_column_values_to_be_between",
    column="price", min_value=50000, max_value=300000000, mostly=0.95
)

add("expect_column_values_to_be_in_set",
    column="transmission",
    value_set=["Automatic", "Manual"],
    mostly=0.95
)

add("expect_column_values_to_be_in_set",
    column="fuel",
    value_set=["Petrol", "Diesel", "Hybrid", "CNG", "Electric", "LPG"],
    mostly=0.90
)

add("expect_column_values_to_be_in_set",
    column="body",
    value_set=["Sedan","Hatchback","SUV","Crossover","Pickup","Van","Coupe","Wagon","Truck","Mini Van"],
    mostly=0.85
)

In [19]:
context.update_expectation_suite(expectation_suite=suite)
print(f"Expectation Suite '{SUITE_NAME}' saved.")

Expectation Suite 'PakWheels_Real_World_Validation' saved.


In [20]:
# Using datasource created in Task 1,2
datasource = context.get_datasource("pakwheels_pandas")

ASSET_NAME = "real_pakwheels_asset"

try:
    asset = datasource.get_asset(ASSET_NAME)
except Exception:
    asset = datasource.add_dataframe_asset(name=ASSET_NAME)

batch_request = asset.build_batch_request(dataframe=df_pak)

validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite_name=SUITE_NAME
)

print("Validator initialized successfully.")

Validator initialized successfully.


In [21]:
results = validator.validate()

stats = results["statistics"]
print(f"Success Rate: {stats['success_percent']:.2f}%")
print(f"Passed: {stats['successful_expectations']}")
print(f"Failed: {stats['unsuccessful_expectations']}")

context.build_data_docs()
print("Data Docs updated. You can find them in gx/uncommitted/data_docs/local_site/index.html")

# Optional: Display failed expectations directly in the notebook
if not results["success"]:
    print("\nDetailed Failures:")
    for res in results["results"]:
        if not res["success"]:
            print(f"- {res['expectation_config']['expectation_type']} failed for column: {res['expectation_config']['kwargs'].get('column')}")

Calculating Metrics:   0%|          | 0/50 [00:00<?, ?it/s]

Success Rate: 94.12%
Passed: 16
Failed: 1
Data Docs updated. You can find them in gx/uncommitted/data_docs/local_site/index.html

Detailed Failures:
- expect_column_values_to_not_be_null failed for column: year


## Task 4: Data Cleaning and Re-validation

In [22]:
import pandas as pd
import numpy as np

print('=' * 60)
print('CORRUPTED SYNTHETIC DATASET - Pre-cleaning Audit')
print('=' * 60)
print(f'\nShape: {corrupted_df.shape[0]:,} rows x {corrupted_df.shape[1]} columns')

print('\n--- NULL counts ---')
nulls = corrupted_df.isnull().sum()
print(nulls[nulls > 0].to_string())

print(f'\n--- Duplicate IDs: {corrupted_df["id"].duplicated().sum()} ---')

bad_years = corrupted_df[~corrupted_df['year'].between(1990, 2025)]
print(f'\n--- Invalid years (outside 1990-2025): {len(bad_years)} rows ---')
print(bad_years['year'].value_counts().to_string())
print(f'\n--- Negative prices: {(corrupted_df["price_pkr"] < 0).sum()} rows ---')

bad_eng = corrupted_df[corrupted_df['engine_cc'].notna() & ~corrupted_df['engine_cc'].between(500, 5000)]
print(f'\n--- Out-of-range engine_cc: {len(bad_eng)} rows ---')
print('\n--- transmission unique values ---')
print(corrupted_df['transmission'].value_counts().to_string())

print('\n--- fuel_type unique values ---')
print(corrupted_df['fuel_type'].value_counts().to_string())

CORRUPTED SYNTHETIC DATASET - Pre-cleaning Audit

Shape: 50,500 rows x 14 columns

--- NULL counts ---
engine_cc     3951
mileage_km    3015
city          2011

--- Duplicate IDs: 500 ---

--- Invalid years (outside 1990-2025): 2037 rows ---
year
1950    530
1800    512
2035    509
2099    486

--- Negative prices: 1530 rows ---

--- Out-of-range engine_cc: 1528 rows ---

--- transmission unique values ---
transmission
Automatic    23949
Manual       23923
Manuall        660
AUTO           659
autmatic       657
MANUAL         652

--- fuel_type unique values ---
fuel_type
Hybrid      12050
Petrol      11986
CNG         11969
Diesel      11953
Gazoline      729
deisel        615
PETROL        609
Gas           589


In [23]:
print('REAL PAKWHEELS DATASET - Pre-cleaning Audit')
print('=' * 60)
print(f'\nShape: {df_pak.shape[0]:,} rows x {df_pak.shape[1]} columns')

print('\n--- NULL counts ---')
nulls_pak = df_pak.isnull().sum()
print(nulls_pak[nulls_pak > 0].to_string())

print(f'\n--- year nulls specifically: {df_pak["year"].isnull().sum()} ---')
print(f'\n--- Duplicate addref: {df_pak["addref"].duplicated().sum()} ---')

print('\n--- transmission unique values ---')
print(df_pak['transmission'].value_counts().to_string())

print('\n--- fuel unique values ---')
print(df_pak['fuel'].value_counts().to_string())

REAL PAKWHEELS DATASET - Pre-cleaning Audit

Shape: 62,302 rows x 14 columns

--- NULL counts ---
assembly    42928
body         7091
year         3786
engine          3
fuel          709
color        1192
price         472

--- year nulls specifically: 3786 ---

--- Duplicate addref: 0 ---

--- transmission unique values ---
transmission
Automatic    34270
Manual       28032

--- fuel unique values ---
fuel
Petrol    56550
Diesel     2732
Hybrid     2311


In [24]:
import pandas as pd
import numpy as np

def clean_synthetic(df):
    df = df.copy()
    original_len = len(df)
    print(f'  Starting rows: {original_len:,}')

    # Remove duplicate rows 
    df = df.drop_duplicates(subset=['id'], keep='first')
    print(f'  After dedup: {len(df):,} rows (removed {original_len - len(df):,})')

    #  invalid years, replace with median of valid years
    valid_year_mask = df['year'].between(1990, 2025)
    median_year     = int(df.loc[valid_year_mask, 'year'].median())
    bad_count       = (~valid_year_mask).sum()
    df.loc[~valid_year_mask, 'year'] = median_year
    print(f'  Fixed {bad_count:,} invalid years -> replaced with median {median_year}')

    # -fix negative prices 
    neg_mask  = df['price_pkr'] < 0
    neg_count = neg_mask.sum()
    df.loc[neg_mask, 'price_pkr'] = df.loc[neg_mask, 'price_pkr'].abs()
    print(f'  Fixed {neg_count:,} negative prices -> abs() applied')

    # clip out-of-range engine_cc 
    # Do this BEFORE imputing nulls so the median is clean.
    bad_eng_mask  = df['engine_cc'].notna() & ~df['engine_cc'].between(500, 5000)
    bad_eng_count = bad_eng_mask.sum()
    df.loc[bad_eng_mask, 'engine_cc'] = df.loc[bad_eng_mask, 'engine_cc'].clip(500, 5000)
    print(f'  Clipped {bad_eng_count:,} out-of-range engine_cc values to [500, 5000]')

    # Impute NULL numerics with median
    for col in ['engine_cc', 'mileage_km']:
        null_count = df[col].isnull().sum()
        if null_count > 0:
            col_median = df[col].median()
            df[col]    = df[col].fillna(col_median)
            print(f'  Imputed {null_count:,} NULLs in "{col}" with median {col_median:.0f}')

    # Impute NULL city with mode 
    city_null = df['city'].isnull().sum()
    if city_null > 0:
        city_mode = df['city'].mode()[0]
        df['city'] = df['city'].fillna(city_mode)
        print(f'  Imputed {city_null:,} NULLs in "city" with mode "{city_mode}"')

    # Remap misspelled transmission values 

    trans_remap = {
        'Automatic': 'Automatic', 'Manual': 'Manual',
        'Manuall':   'Manual',    'Auto':   'Automatic',
        'Autmatic':  'Automatic', 'Automtic':'Automatic',
        'Manaul':    'Manual',
    }
    df['transmission'] = df['transmission'].str.strip().str.title().replace(trans_remap)

    valid_trans = {'Automatic', 'Manual'}
    still_bad   = ~df['transmission'].isin(valid_trans)
    if still_bad.sum() > 0:
        mode_val = df.loc[~still_bad, 'transmission'].mode()[0]
        df.loc[still_bad, 'transmission'] = mode_val
        print(f'  Catch-all: replaced {still_bad.sum()} unrecognised transmission values')
    print(f'  transmission cleaned -> {sorted(df["transmission"].unique())}')

    # remap misspelled fuel_type values
    fuel_remap = {
        'Petrol':   'Petrol',   'Diesel':  'Diesel',
        'Hybrid':   'Hybrid',   'Cng':     'CNG',
        'Gazoline': 'Petrol',   'Deisel':  'Diesel',
        'Gas':      'CNG',      'Lng':     'LPG',
    }
    df['fuel_type'] = df['fuel_type'].str.strip().str.title().replace(fuel_remap)
    valid_fuel  = {'Petrol', 'Diesel', 'Hybrid', 'CNG'}
    still_bad_f = ~df['fuel_type'].isin(valid_fuel)
    if still_bad_f.sum() > 0:
        fuel_mode = df.loc[~still_bad_f, 'fuel_type'].mode()[0]
        df.loc[still_bad_f, 'fuel_type'] = fuel_mode
        print(f'  Catch-all: replaced {still_bad_f.sum()} unrecognised fuel_type values')
    print(f'  fuel_type cleaned -> {sorted(df["fuel_type"].unique())}')

    # Standardise remaining string columns 
    for col in ['make', 'model', 'city', 'body_type', 'color']:
        if col in df.columns:
            df[col] = df[col].str.strip().str.title()

    # Fix dtypes 
    # After fillna, numeric cols become float64. Cast back to int.
    for col in ['engine_cc', 'mileage_km', 'year', 'num_owners']:
        if col in df.columns:
            df[col] = df[col].round().astype(int)

    print(f'\n  Final shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
    return df


print('Cleaning corrupted synthetic dataset...')
print('-' * 40)
cleaned_synthetic = clean_synthetic(corrupted_df)
print('\n Synthetic dataset cleaned.')

Cleaning corrupted synthetic dataset...
----------------------------------------
  Starting rows: 50,500
  After dedup: 50,000 rows (removed 500)
  Fixed 2,016 invalid years -> replaced with median 2014
  Fixed 1,508 negative prices -> abs() applied
  Clipped 1,512 out-of-range engine_cc values to [500, 5000]
  Imputed 3,906 NULLs in "engine_cc" with median 1500
  Imputed 2,987 NULLs in "mileage_km" with median 104560
  Imputed 1,988 NULLs in "city" with mode "Karachi"
  transmission cleaned -> ['Automatic', 'Manual']
  fuel_type cleaned -> ['CNG', 'Diesel', 'Hybrid', 'Petrol']

  Final shape: 50,000 rows x 14 columns

 Synthetic dataset cleaned.


In [25]:
def clean_real_pakwheels(df):
    df = df.copy()
    original_len = len(df)
    print(f'  Starting rows: {original_len:,}')

    numeric_cols = ['year', 'engine', 'mileage', 'price']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = (
                df[col].astype(str)
                       .str.replace(',', '', regex=False)
                       .str.replace(' ', '', regex=False)
            )
            df[col] = pd.to_numeric(df[col], errors='coerce')
    print('  Numeric columns coerced to float.')

    #  Impute NULL year 
    year_null = df['year'].isnull().sum()
    if year_null > 0:
        valid_yr   = df['year'].between(1990, 2026)
        median_yr  = int(df.loc[valid_yr, 'year'].median())
        df['year'] = df['year'].fillna(median_yr)
        print(f'  Imputed {year_null:,} NULL years with median {median_yr}')

    # Remove rows with impossible year values 
    before = len(df)
    df = df[df['year'].between(1990, 2026)]
    print(f'  Removed {before - len(df):,} rows with year outside [1990, 2026]')

    #  Remove rows with impossible engine/price/mileage 
    for col, lo, hi in [('engine', 500, 6000),
                         ('price', 50000, 300000000),
                         ('mileage', 0, 500000)]:
        if col in df.columns:
            before = len(df)
            df = df[df[col].isna() | df[col].between(lo, hi)]
            print(f'  Removed {before - len(df):,} rows with {col} outside [{lo}, {hi}]')

    # Impute remaining numeric NULLs with median 
    for col in ['engine', 'mileage', 'price']:
        if col in df.columns:
            null_count = df[col].isnull().sum()
            if null_count > 0:
                med = df[col].median()
                df[col] = df[col].fillna(med)
                print(f'  Imputed {null_count:,} NULLs in "{col}" with median {med:.0f}')

    if 'transmission' in df.columns:
        trans_remap = {
            'Automatic':'Automatic','Manual':'Manual',
            'Auto':'Automatic','Manuall':'Manual','Autmatic':'Automatic',
        }
        df['transmission'] = (
            df['transmission'].astype(str).str.strip().str.title().replace(trans_remap)
        )
        valid_trans = {'Automatic', 'Manual'}
        mask = ~df['transmission'].isin(valid_trans)
        if mask.sum() > 0:
            mode_v = df.loc[~mask, 'transmission'].mode()[0]
            df.loc[mask, 'transmission'] = mode_v
        print(f'  transmission -> {sorted(df["transmission"].unique())}')

    # Standardise fuel values
    if 'fuel' in df.columns:
        fuel_remap = {
            'Petrol':'Petrol','Diesel':'Diesel','Hybrid':'Hybrid',
            'Cng':'CNG','Electric':'Electric','Lpg':'LPG',
            'Gazoline':'Petrol','Deisel':'Diesel','Gas':'CNG',
        }
        df['fuel'] = df['fuel'].fillna('Unknown')
        df['fuel'] = df['fuel'].astype(str).str.strip().str.title().replace(fuel_remap)
        df = df[df['fuel'] != 'Unknown']
        print(f'  fuel -> {sorted(df["fuel"].unique())}')

    #  duplicate addref 
    if 'addref' in df.columns:
        before = len(df)
        df = df.drop_duplicates(subset=['addref'], keep='first')
        if before - len(df) > 0:
            print(f'  Removed {before - len(df):,} duplicate addref rows')

    for col in ['city', 'make', 'model', 'body', 'color']:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.title()
    df['year'] = df['year'].round().astype(int)

    print(f'\n  Final shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
    return df


print('Cleaning real PakWheels dataset...')
print('-' * 40)
cleaned_pakwheels = clean_real_pakwheels(df_pak)
print('\n Real PakWheels dataset cleaned.')

Cleaning real PakWheels dataset...
----------------------------------------
  Starting rows: 62,302
  Numeric columns coerced to float.
  Imputed 3,786 NULL years with median 2015
  Removed 0 rows with year outside [1990, 2026]
  Removed 195 rows with engine outside [500, 6000]
  Removed 0 rows with price outside [50000, 300000000]
  Removed 371 rows with mileage outside [0, 500000]
  Imputed 3 NULLs in "engine" with median 1300
  Imputed 464 NULLs in "price" with median 2700000
  transmission -> ['Automatic', 'Manual']
  fuel -> ['Diesel', 'Hybrid', 'Petrol']

  Final shape: 61,158 rows x 14 columns

 Real PakWheels dataset cleaned.


In [26]:
import pandas as pd

print('=' * 55)
print('SYNTHETIC DATASET - Before vs After')
print('=' * 55)

comparison_synth = pd.DataFrame({
    'Issue': [
        'Total NULLs', 'Duplicate IDs', 'Invalid years',
        'Negative prices', 'Invalid transmission', 'Invalid fuel_type'
    ],
    'Before': [
        corrupted_df.isnull().sum().sum(),
        corrupted_df['id'].duplicated().sum(),
        (~corrupted_df['year'].between(1990, 2025)).sum(),
        (corrupted_df['price_pkr'] < 0).sum(),
        (~corrupted_df['transmission'].isin(['Automatic','Manual'])).sum(),
        (~corrupted_df['fuel_type'].isin(['Petrol','Diesel','Hybrid','CNG'])).sum(),
    ],
    'After': [
        cleaned_synthetic.isnull().sum().sum(),
        cleaned_synthetic['id'].duplicated().sum(),
        (~cleaned_synthetic['year'].between(1990, 2025)).sum(),
        (cleaned_synthetic['price_pkr'] < 0).sum(),
        (~cleaned_synthetic['transmission'].isin(['Automatic','Manual'])).sum(),
        (~cleaned_synthetic['fuel_type'].isin(['Petrol','Diesel','Hybrid','CNG'])).sum(),
    ],
})
comparison_synth['Fixed?'] = comparison_synth.apply(
    lambda r: 'YES' if r['After'] == 0 else ('REDUCED' if r['After'] < r['Before'] else 'NO'),
    axis=1
)
display(comparison_synth)

print('\n')
print('=' * 55)
print('REAL PAKWHEELS - Before vs After')
print('=' * 55)

comparison_real = pd.DataFrame({
    'Issue': ['NULL year values', 'Total rows'],
    'Before': [df_pak['year'].isnull().sum(), len(df_pak)],
    'After':  [cleaned_pakwheels['year'].isnull().sum(), len(cleaned_pakwheels)],
})
display(comparison_real)

SYNTHETIC DATASET - Before vs After


,Issue,Before,After,Fixed?
0,Total NULLs,8977,0,YES
1,Duplicate IDs,500,0,YES
2,Invalid years,2037,0,YES
3,Negative prices,1530,0,YES
4,Invalid transmission,2628,0,YES
5,Invalid fuel_type,2542,0,YES




REAL PAKWHEELS - Before vs After


,Issue,Before,After
0,NULL year values,3786,0
1,Total rows,62302,61158
